# 04 - ARFIMA: Memoria Longa e Diferenciacao Fracionaria

Este notebook apresenta o modelo **ARFIMA(p,d,q)** para series com memoria longa.

**Conteudo:**
1. Conceito de memoria longa
2. Diferenciacao fracionaria
3. Estimacao do parametro d
4. ARFIMA com chronobox
5. Comparacao ARFIMA vs ARIMA
6. Exercicios

## 1. Memoria Longa: Conceito

Uma serie temporal tem **memoria longa** quando suas autocorrelacoes decaem lentamente (hiperbolicamente), em contraste com o decaimento exponencial de processos ARMA.

**Formalmente:** Um processo estacionario $y_t$ tem memoria longa se:
$$\rho(k) \sim C \cdot k^{2d-1} \quad \text{quando } k \to \infty$$

onde $0 < d < 0.5$ e o parametro de memoria longa.

**Interpretacao de $d$:**
- $d = 0$: memoria curta (ARMA classico)
- $0 < d < 0.5$: memoria longa estacionaria — autocorrelacoes decaem lentamente
- $d = 0.5$: fronteira da nao-estacionariedade
- $d = 1$: raiz unitaria (ARIMA com diferenciacao inteira)

**O problema do ARIMA:** forcar $d \in \{0, 1\}$ pode ser sub- ou sobre-diferenciacao.

## 2. Diferenciacao Fracionaria

A diferenciacao fracionaria generaliza $(1-L)^d$ para $d$ real:

$$(1-L)^d = \sum_{k=0}^{\infty} \binom{d}{k} (-L)^k = 1 - dL + \frac{d(d-1)}{2!}L^2 - \frac{d(d-1)(d-2)}{3!}L^3 + \cdots$$

Os coeficientes $\pi_k$ da expansao sao:
$$\pi_k = \frac{\Gamma(k-d)}{\Gamma(k+1)\Gamma(-d)}$$

O modelo ARFIMA(p,d,q) e:
$$\phi(L)(1-L)^d y_t = \theta(L) \varepsilon_t, \quad d \in (-0.5, 0.5)$$

## 3. Setup e Dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox import ARIMA, ARFIMA
from chronobox.models.arfima import (
    estimate_d_gph, estimate_d_local_whittle,
    fractional_diff, fractional_diff_coefficients,
    simulate_arfima
)
from chronobox.tests_stat import adf_test, kpss_test, ljung_box_test
from chronobox.visualization import plot_diagnostics, set_theme

set_theme('professional')
np.random.seed(42)

print('chronobox importado com sucesso!')

In [ ]:
import os
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

# Carregar Nile (classico para memoria longa)
nile = pd.read_csv(os.path.join(DATA_DIR, 'nile.csv'), parse_dates=['date'])
nile.set_index('date', inplace=True)
y_nile = nile['flow'].values

# Carregar IPCA
ipca = pd.read_csv(os.path.join(DATA_DIR, 'brazil_ipca.csv'), parse_dates=['date'])
ipca.set_index('date', inplace=True)
y_ipca = ipca['ipca'].values

print(f'Nile: {len(y_nile)} obs')
print(f'IPCA: {len(y_ipca)} obs')

## 4. Simulacao de Processo com Memoria Longa

Vamos primeiro simular um ARFIMA para visualizar o efeito de diferentes valores de $d$.

In [ ]:
# Grafico 1: Efeito de d na serie temporal
rng = np.random.default_rng(42)
n = 500

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
d_values = [0.0, 0.15, 0.30, 0.45]

for ax, d_val in zip(axes.flat, d_values):
    y_sim = simulate_arfima(n=n, d=d_val, sigma=1.0, rng=rng)
    ax.plot(y_sim, color='steelblue', linewidth=0.8)
    ax.set_title(f'd = {d_val:.2f}')
    ax.set_xlabel('t')

plt.suptitle('Simulacoes ARFIMA(0,d,0) para diferentes valores de d', fontsize=13)
plt.tight_layout()
plt.show()

print('Conforme d aumenta: maior persistencia, tendencias locais mais pronunciadas.')
print('d=0: ruido branco. d proximo de 0.5: quase nao-estacionario.')

In [ ]:
# Coeficientes de diferenciacao fracionaria
fig, ax = plt.subplots(figsize=(10, 5))

for d_val in [0.1, 0.2, 0.3, 0.4]:
    coefs = fractional_diff_coefficients(d_val, 30)
    ax.plot(range(len(coefs)), coefs, marker='o', markersize=4, label=f'd={d_val}')

ax.set_xlabel('k')
ax.set_ylabel('Coeficiente pi_k')
ax.set_title('Coeficientes de Diferenciacao Fracionaria (1-L)^d')
ax.legend()
ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Estimacao do Parametro d

Dois metodos classicos para estimar $d$:

1. **GPH** (Geweke & Porter-Hudak, 1983): regressao log-periodograma
2. **Local Whittle** (Robinson, 1995): estimador semi-parametrico

Ambos sao metodos de dominio da frequencia.

In [ ]:
# Estimacao de d no Nile
d_gph, se_gph = estimate_d_gph(y_nile)
d_whittle, se_whittle = estimate_d_local_whittle(y_nile)

print('=== Estimacao de d para o Nile ===')
print(f'GPH:           d = {d_gph:.4f} (se = {se_gph:.4f})')
print(f'Local Whittle: d = {d_whittle:.4f} (se = {se_whittle:.4f})')
print()

# Estimacao de d no IPCA
d_gph_ipca, se_gph_ipca = estimate_d_gph(y_ipca)
d_whittle_ipca, se_whittle_ipca = estimate_d_local_whittle(y_ipca)

print('=== Estimacao de d para o IPCA ===')
print(f'GPH:           d = {d_gph_ipca:.4f} (se = {se_gph_ipca:.4f})')
print(f'Local Whittle: d = {d_whittle_ipca:.4f} (se = {se_whittle_ipca:.4f})')

print()
print('Se d > 0 significativamente, a serie possui memoria longa.')
print('Se d proximo de 0.5, considere que pode ser nao-estacionaria.')

## 6. Ajuste ARFIMA com chronobox

In [ ]:
# ARFIMA com d estimado conjuntamente no Nile
arfima_model = ARFIMA(order=(1, 0.0, 0))
arfima_results = arfima_model.fit(y_nile, estimate_d=True)

print(arfima_results.summary())

In [ ]:
# Grafico 3: Diagnostico ARFIMA
fig = plot_diagnostics(residuals=arfima_results.resid, lags=15,
                       title=f'Diagnostico - ARFIMA({arfima_results.order[0]}, {arfima_results.d:.3f}, {arfima_results.order[2]})')
plt.show()

In [ ]:
# Diferentes especificacoes ARFIMA
especificacoes = [
    (0, 0),  # ARFIMA(0,d,0) - memoria longa pura
    (1, 0),  # ARFIMA(1,d,0)
    (0, 1),  # ARFIMA(0,d,1)
    (1, 1),  # ARFIMA(1,d,1)
]

print(f'{"Modelo":<22} {"d":>8} {"AIC":>10} {"BIC":>10}')
print('-' * 52)
arfima_fits = {}
for p, q in especificacoes:
    m = ARFIMA(order=(p, 0.0, q))
    r = m.fit(y_nile, estimate_d=True)
    nome = f'ARFIMA({p},{r.d:.3f},{q})'
    arfima_fits[nome] = r
    print(f'{nome:<22} {r.d:>8.4f} {r.aic:>10.2f} {r.bic:>10.2f}')

melhor = min(arfima_fits, key=lambda k: arfima_fits[k].aic)
print(f'\nMelhor ARFIMA por AIC: {melhor}')

## 7. Comparacao: ARFIMA vs ARIMA

Vamos comparar ARFIMA (diferenciacao fracionaria) com ARIMA (diferenciacao inteira).

In [ ]:
# ARIMA com d=0 e d=1
arima_d0 = ARIMA(order=(1, 0, 1))
res_d0 = arima_d0.fit(y_nile)

arima_d1 = ARIMA(order=(1, 1, 1))
res_d1 = arima_d1.fit(y_nile)

# ARFIMA com d estimado
arfima_best = ARFIMA(order=(1, 0.0, 1))
res_arfima = arfima_best.fit(y_nile, estimate_d=True)

print(f'{"Modelo":<30} {"d":>8} {"AIC":>10} {"BIC":>10}')
print('-' * 60)
print(f'{"ARIMA(1,0,1) [d=0]":<30} {"0":>8} {res_d0.aic:>10.2f} {res_d0.bic:>10.2f}')
print(f'{"ARIMA(1,1,1) [d=1]":<30} {"1":>8} {res_d1.aic:>10.2f} {res_d1.bic:>10.2f}')
print(f'{"ARFIMA(1,d,1) [d estimado]":<30} {res_arfima.d:>8.4f} {res_arfima.aic:>10.2f} {res_arfima.bic:>10.2f}')

print()
print('O ARFIMA permite d fracionario, potencialmente capturando melhor')
print('a dinamica de series com persistencia intermediaria entre I(0) e I(1).')

In [ ]:
# Grafico 4: Previsao ARFIMA vs ARIMA
steps = 15
fc_arima = res_d1.forecast(steps=steps)
fc_arfima = res_arfima.forecast(steps=steps)

n = len(y_nile)
fig, ax = plt.subplots(figsize=(12, 5))

# Ultimos 40 pontos + previsao
ax.plot(range(n-40, n), y_nile[-40:], color='steelblue', linewidth=1.5, label='Observado')
ax.plot(range(n, n+steps), fc_arima['forecast'], color='red', linewidth=2, 
        linestyle='--', marker='s', markersize=4, label='ARIMA(1,1,1)')
ax.fill_between(range(n, n+steps), fc_arima['lower'], fc_arima['upper'], alpha=0.15, color='red')
ax.plot(range(n, n+steps), fc_arfima, color='darkorange', linewidth=2, 
        linestyle='-', marker='o', markersize=4, label=f'ARFIMA(1,{res_arfima.d:.2f},1)')
ax.axvline(n-1, color='gray', linestyle=':', alpha=0.5)
ax.set_title('Comparacao de Previsoes: ARIMA vs ARFIMA')
ax.set_xlabel('Observacao')
ax.set_ylabel('Vazao')
ax.legend()
plt.tight_layout()
plt.show()

print('O ARFIMA tende a reverter mais rapidamente para a media de longo prazo,')
print('enquanto o ARIMA(d=1) projeta previsoes mais planas (random walk).')

In [ ]:
# Grafico 5: ACF dos residuos - ARIMA vs ARFIMA
from chronobox.visualization.diagnostics_plot import _compute_acf

max_lag = 20
acf_arima = _compute_acf(res_d1.residuals, max_lag)
acf_arfima = _compute_acf(res_arfima.resid, max_lag)
ci = 1.96 / np.sqrt(len(y_nile))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(max_lag + 1), acf_arima, width=0.4, color='red', alpha=0.7)
axes[0].axhline(ci, color='black', linestyle='--', linewidth=0.7)
axes[0].axhline(-ci, color='black', linestyle='--', linewidth=0.7)
axes[0].set_title('ACF Residuos - ARIMA(1,1,1)')
axes[0].set_xlabel('Lag')

axes[1].bar(range(max_lag + 1), acf_arfima, width=0.4, color='darkorange', alpha=0.7)
axes[1].axhline(ci, color='black', linestyle='--', linewidth=0.7)
axes[1].axhline(-ci, color='black', linestyle='--', linewidth=0.7)
axes[1].set_title(f'ACF Residuos - ARFIMA(1,{res_arfima.d:.2f},1)')
axes[1].set_xlabel('Lag')

plt.suptitle('Comparacao da ACF dos Residuos', fontsize=13)
plt.tight_layout()
plt.show()

print('Se o ARFIMA captura a memoria longa corretamente,')
print('seus residuos devem ter ACF mais proxima de zero em todos os lags.')

## Exercicio

### Exercicio 1: Detectar memoria longa no brazil_ipca.csv

1. Carregue o dataset `brazil_ipca.csv`
2. Estime $d$ usando os metodos GPH e Local Whittle
3. A inflacao brasileira apresenta memoria longa?
4. Ajuste um ARFIMA e compare com ARIMA(1,1,1)
5. Qual modelo se ajusta melhor aos dados?

**Contexto economico:** A persistencia inflacionaria e um fenomeno bem documentado — a teoria economica sugere que expectativas de inflacao geram inercial, o que pode se manifestar como memoria longa.

In [ ]:
# Exercicio 1 - Seu codigo aqui
# ...

## Exercicio

### Exercicio 2: Simulacao Monte Carlo

1. Simule 200 series ARFIMA(0,0.3,0) com n=300 usando `simulate_arfima`
2. Para cada serie, estime $d$ via GPH e Local Whittle
3. Plote a distribuicao das estimativas de $d$
4. Qual metodo tem menor vies e menor variancia?

In [ ]:
# Exercicio 2 - Seu codigo aqui
# ...